# Step 3 - Make the baseline corpus (E0)

We take real training questions, let the untrained model reason through each
one, check if it got the answer right, and score the reasoning with the
argument pipeline. Everything is saved to one file.

This is our raw material. The correlation study reads it to ask: do the
argument scores line up with getting the answer right? It is also a finding on
its own: how does the model reason before any training.

No training here, just generation and scoring. Run cells top to bottom.

## Setup: get the repo (force latest code)

In [1]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
subprocess.run(["git", "fetch", "--quiet"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready, at", os.getcwd())

repo ready, at /content/rlvr-argument-mining


## Install what we need

Generation needs transformers and bitsandbytes. Scoring needs networkx. These are usually already on Colab.

In [2]:
!pip install -q transformers bitsandbytes accelerate datasets networkx pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 72.6 MB/s eta 0:00:00


## Check the GPU is on

Should print True. If it says False, use Runtime > Change runtime type > GPU (L4), then run from the top.

In [3]:
import torch
print("gpu:", torch.cuda.is_available())

gpu: True


## Small test first: 5 questions

Before the full run, do 5 questions to check the whole thing works: the model
generates, the answer is read, the argument scores compute, and it saves. This
takes a couple of minutes (it downloads the model the first time).

In [ ]:
from e0_corpus import generate

generate(n_questions=5)

README.md:   0%|          | 0.00/9.62k [00:00<?, ?B/s]

formal_fallacies/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 34.6kB            

formal_fallacies/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

logical_deduction_three_objects/test-000(…): reconstructing file:   0%|          |  0.00B / 20.6kB            

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## The full run: 200 questions

If the test looked right, run the real corpus. This is slow because the model
writes a full reasoning trace for each question, one at a time. Expect roughly
30 to 60 minutes for 200.

It is safe to stop and re-run. The saver skips questions already done, so if the
session dies you just run this cell again and it carries on.

In [ ]:
from e0_corpus import generate

generate(n_questions=200)

## Look at what we made

Prints how many the model got right, how often answer-reading failed, and the
average argument scores. This is the first real look at how the untrained model
reasons.

In [ ]:
from e0_corpus import summarise

summarise()

## What to do next

- If the numbers look sensible: tell Claude and we build the correlation study
  (does argument structure predict correct answers?).
- If something looks off (all wrong, all zeros, lots of extraction failures):
  paste the output back.